# ECB API & Norges Bank API - bronze
Valutakurser fra 2 API, lagret i hver sin tabell

Før bruk bør det sjekkes om systemet som skal bruke valutakurser for dagen før. Kursene settes som regel på ettermiddagen, og noen systemer vil da helst ha en forskyvning i kursdatoene. Koden under tar ikke hensyn til dette.

## Bibliotek

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TestApp").master("local[*]").getOrCreate()

In [ ]:
from datetime import datetime
from datetime import timedelta
import io
import pandas as pd
from pyspark.sql import Window
import pyspark.sql.functions as F
import pyspark.sql.types as T
import requests

## Variabler

In [ ]:
periode_start = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
periode_slutt = datetime.now().strftime("%Y-%m-%d")

valuta = ["DKK", "EUR","GBP","ISK", "PLN","SEK","USD"]
valuta_nb = "+".join(valuta)

valuta.append("NOK")
valuta.remove("EUR")
name_ecb = [f"{item} to EUR" for item in valuta]
pair_ecb = [f"{item}.EUR" for item in valuta]
valuta_ecb = dict(zip(name_ecb, pair_ecb))

base_url_ecb = "https://data-api.ecb.europa.eu/service/data/EXR/D.{pair}.SP00.A?format=csvdata&startPeriod={start}&endPeriod={end}"
base_url_nb = f"https://data.norges-bank.no/api/data/EXR/B.{valuta_nb}.NOK.SP?format=csv&startPeriod={periode_start}&endPeriod={periode_slutt}&locale=en"

## Kode

In [ ]:
# norges bank api
try: 
    response = requests.get(base_url_nb)
    df_nb = pd.read_csv(io.BytesIO(response.content), sep = ";")
    df_nb.rename(columns={"Tenor": "Tenor_2"}, inplace=True)
    df_nb = spark.createDataFrame(df_nb) 
    
except Exception as e:
    print(f"Got en error while reading the NB API to dataframe: {e}. The API returned a response code of {response.status_code}")
    df_nb = None

In [ ]:
# ecb api
df_ecb = None

for name, pair in valuta_ecb.items():

    url = base_url_ecb.format(pair=pair, start=periode_start, end=periode_slutt)
    response = requests.get(url)

    # les med pandas og bruk bytesio som buffer 
    temp_ecb = pd.read_csv(io.BytesIO(response.content), sep=",")

    # hjelpekolonner
    temp_ecb["conversion"] = name
    temp_ecb["pair_code"] = pair

    temp_ecb = spark.createDataFrame(temp_ecb) 

    # hvis spark df ikke er None, append
    df_ecb = temp_ecb if df_ecb is None else df_ecb.unionByName(temp_ecb)

# Silver
Bruker data fra ECB og NB, setter sammen til 1 tabell.

In [ ]:
# ECB
# lag id_dato and id_aarmnd fra timeperiod

# first converting the time_period to date
df_ecb = df_ecb.withColumn("time_period", F.to_date(F.col("time_period"), "yyyy-MM-dd")
)

# creating the id_columns
df_ecb = df_ecb.withColumn("id_dato", F.date_format(F.col("time_period"), "yyyyMMdd")
    ).withColumn("id_aarmnd", F.date_format(F.col("time_period"), "yyyyMM")
)

df_ecb = df_ecb.withColumn("source_api", F.lit("ECB"))

# dict to rename columns, also used for ordering, order matters
cols_to_rename_ecb = {
"id_dato": "id_dato",
"id_aarmnd": "id_aarmnd",
"currency": "base_currency",
"currency_denom": "to_currency",
"obs_value": "conversion_rate",
"source_api": "source_api"
}

df_ecb = df_ecb.withColumnsRenamed(cols_to_rename_ecb)

col_order_ecb = [new_col for old_col, new_col in cols_to_rename_ecb.items()]

df_ecb = df_ecb.select(*col_order_ecb)

In [ ]:
# NB
# lag id_dato and id_aarmnd fra timeperiod

df_nb = df_nb.withColumn("time_period", F.to_date(F.col("time_period"), "yyyy-MM-dd")
)

df_nb = df_nb.withColumn("id_dato", F.date_format(F.col("time_period"), "yyyyMMdd")
    ).withColumn("id_aarmnd", F.date_format(F.col("time_period"), "yyyyMM")
)

# nb har "i 100" for enkelte valuta. 
df_nb = df_nb.withColumn("obs_value_adjusted",
 F.round(F.when(F.col("unit multiplier") == "Hundreds", F.col("obs_value") / 100.0)
 .otherwise(F.col("obs_value")), 4 # antall desimaler
)
)

df_nb = df_nb.withColumn("source_api", F.lit("NorgesBank"))

cols_to_rename_nb = {
    "id_dato": "id_dato",
    "id_aarmnd": "id_aarmnd",
    "base_cur": "base_currency",
    "quote_cur": "to_currency",
    "obs_value_adjusted": "conversion_rate",
    "source_api": "source_api"
}

df_nb = df_nb.withColumnsRenamed(cols_to_rename_nb)

col_order_nb = [new_col for old_col, new_col in cols_to_rename_nb.items()]

df_nb = df_nb.select(*col_order_nb)

# union ecb og nb
df_base = df_ecb.unionByName(df_nb)

# Gold
Lager 2 tabeller for valuta, 1 med daglig kurser og 1 med månedlige snitt.

In [ ]:
## Daglige kurser

# noen justeringer: dropp id_aarmnd og fyll inn manglende datoer med forrige dags verdi.
# dette må gjøres kun på den daglige tabellen, for å ikke påvirke tabellen for månedlige kurser.

df_daily = df_base.drop(F.col("id_aarmnd"))

# Identifiser manglende datoer og fyll dem inn
df_daily = df_daily.withColumn("id_dato", F.to_date("id_dato", "yyyyMMdd"))

min_date, max_date = df_daily.agg(F.min("id_dato"), F.max("id_dato")).first()

date_df = spark.createDataFrame([(min_date, max_date)], ["start", "end"]
).withColumn("id_dato", F.explode(F.sequence(F.col("start"), F.col("end")))
).select("id_dato")

currency_pairs = df_daily.select("base_currency", "to_currency", "source_api").distinct()

# alle datoer og kursene
full_df = date_df.crossJoin(currency_pairs)

# join inn i originale data, manglende datoer har NULL i kursverdi
joined_df = full_df.join(df_daily, ["id_dato", "base_currency", "to_currency", "source_api"], "left")

# bruker window til å fylle inn NULL
window = Window.partitionBy("base_currency", "to_currency", "source_api").orderBy("id_dato")

filled_df = joined_df.withColumn("conversion_rate",F.last("conversion_rate", ignorenulls=True).over(window))

# id_dato har format yyyy-MM-dd, endres tilbake til yyyyMMdd
filled_df = filled_df.withColumn("id_dato", F.date_format(F.col("id_dato"), "yyyyMMdd"))

In [ ]:
## Månedlige kurser

# finner månedlig gjennomsnitt og første og siste kurs.
cols_to_grp_by = ["id_aarmnd", "base_currency", "to_currency", "source_api"]

window = Window.partitionBy(*cols_to_grp_by).orderBy("id_dato")

df_with_first_last = (df_base.withColumn("first_conversion_rate", F.first("conversion_rate").over(window)
 ).withColumn("last_conversion_rate", F.last("conversion_rate").over(window))
)

df_monthly = (df_with_first_last
 .groupBy("id_aarmnd", "base_currency", "to_currency", "source_api")
 .agg(
        F.round(F.avg("conversion_rate"), 4).alias("avg_conversion_rate"),
        F.first("first_conversion_rate").alias("first_conversion_rate"),
        F.last("last_conversion_rate").alias("last_conversion_rate")
 )
)

# hjelper-kolonne for å beregne YTD gjennomsnitt
df_monthly = df_monthly.withColumn("year", F.substring("id_aarmnd", 1, 4)) 

df_monthly = df_monthly.withColumn("id_aarmnd_int", F.col("id_aarmnd").cast("int"))

ytd_window = Window.partitionBy("year", "base_currency", "to_currency", "source_api"
 ).orderBy("id_aarmnd_int"
 ).rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_monthly = df_monthly.withColumn("ytd_avg_conversion_rate",
 F.round(F.avg("avg_conversion_rate").over(ytd_window), 10)
).drop(*["year", "id_aarmnd_int"]
)